# Notebook 04 — Model B: Prediksi Kemacetan 1 Jam & 3 Jam ke Depan
**Proyek:** Sistem Pemantauan & Prediksi Kemacetan Lalu Lintas Kota berbasis IoT + Big Data (Lambda Architecture)

## Posisi notebook ini
```
 BATCH LAYER (nb 01)                                   SERVING LAYER
 master dataset PEMS-BAY ──► fitur lag/profil/spasial/cuaca ──► MODEL B ──► prediksi t+1 jam & t+3 jam ──► dashboard
                                                                            (per sensor, per 15 menit)
```
Model A (notebook 03) menjawab **"seberapa macet sekarang?"**. Model B menjawab pertanyaan yang lebih berharga bagi city planner: **"akan semacet apa 1 dan 3 jam lagi?"** — supaya bisa bertindak *sebelum* macet terjadi (atur lampu, kirim petugas, umumkan rute alternatif).

## Yang akan kamu kuasai
1. **Split kronologis** — di time series, tidak boleh mengacak data (masa depan bocor ke masa lalu).
2. **Feature engineering time series**: lag, rolling window, profil historis, tetangga spasial, cuaca, kalender.
3. **Memprediksi deviasi**, bukan nilai mentah — model fokus pada bagian yang *tidak rutin*.
4. Membandingkan dengan **baseline yang kuat** (persistence & profil historis). Model yang tidak mengalahkannya tidak layak dipakai.
5. Bagaimana logika yang sama ditulis di **Spark** untuk skala cluster.

> Notebook ini **cukup CPU** (tidak perlu GPU). Butuh ±10–15 menit.

## 0. Setup

In [ ]:
import sys, os, shutil, json, time, warnings
IN_COLAB = "google.colab" in sys.modules
%pip install -q lightgbm pyarrow     # pesan "restart the kernel" boleh diabaikan
warnings.filterwarnings("ignore", category=UserWarning)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/Projek Akhir"     # <-- sesuaikan bila perlu
    WORK = "/content/work"
else:
    BASE = os.environ.get("PROJ_BASE", ".")
    WORK = os.environ.get("PROJ_WORK", "./work")

B01        = f"{BASE}/output/01_batch"
MASTER_DIR = f"{B01}/master_pemsbay"
META_CSV   = f"{B01}/ref_sensor_meta.csv"
OUT_DIR    = f"{BASE}/output/04_model_b"
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True); os.makedirs(WORK, exist_ok=True)
for p in [MASTER_DIR, META_CSV]:
    print(("✅ " if os.path.exists(p) else "❌ TIDAK ADA: ") + p)
assert os.path.exists(MASTER_DIR), "Jalankan notebook 01 dulu (master dataset belum ada)"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
def savefig(name): plt.savefig(f"{OUT_DIR}/figures/{name}.png", bbox_inches="tight")

## 1. Membaca master dataset dari batch layer
Kita membaca **master dataset Parquet** hasil notebook 01 — bukan CSV mentah. Inilah gunanya batch layer: semua notebook berikutnya memakai satu sumber kebenaran yang sudah bersih.

Lalu kita ubah ke format **wide** (baris = waktu, kolom = sensor) karena operasi time series (geser/lag, rolling) jauh lebih cepat di matriks NumPy.

In [ ]:
local_master = f"{WORK}/master_pemsbay"
if not os.path.exists(local_master): shutil.copytree(MASTER_DIR, local_master)     # disk lokal lebih cepat dari Drive

m = pd.read_parquet(local_master, columns=["timestamp", "sensor_id", "speed"])
S_df = m.pivot(index="timestamp", columns="sensor_id", values="speed").astype("float32")
S_df = S_df.reindex(pd.date_range(S_df.index.min(), S_df.index.max(), freq="5min"))   # grid waktu reguler
del m
times, sensors, S = S_df.index, S_df.columns.values, S_df.values
T, N = S.shape
meta = pd.read_csv(META_CSV).set_index("sensor_id").loc[sensors]
print(f"Matriks kecepatan: {T:,} langkah waktu × {N} sensor | {times.min()} → {times.max()}")

## 2. Split kronologis
| Bagian | Periode | Fungsi |
|---|---|---|
| **Train** | 1 Jan – 30 Apr 2017 | melatih model, menghitung v_ff & profil |
| **Validasi** | Mei 2017 | *early stopping* (kapan berhenti training) |
| **Test** | Juni 2017 | penilaian akhir — **tidak disentuh** sampai akhir |

⚠️ **Anti-bocor:** *free-flow speed* (v_ff) dan profil historis dihitung **hanya dari data train**. Kalau dihitung dari semua data (seperti analisis di notebook 01), model ikut "melihat" pola bulan Juni — nilai test jadi terlalu bagus dan menipu.

In [ ]:
TRAIN_END, VAL_END = pd.Timestamp("2017-05-01"), pd.Timestamp("2017-06-01")
is_train = np.asarray(times < TRAIN_END)

night = is_train & np.asarray((times.hour >= 22) | (times.hour < 5))
vff = np.nanquantile(S[night], 0.85, axis=0).astype("float32")              # free-flow speed per sensor (train saja)

def slot_of(ts): return np.asarray(ts.dayofweek * 96 + ts.hour * 4 + ts.minute // 15)   # slot 15-menit dalam seminggu (0..671)
SLOT = slot_of(times)
PROFILE = pd.DataFrame(S[is_train]).groupby(SLOT[is_train]).mean().reindex(range(672)).values.astype("float32")
P = PROFILE[SLOT]                                                             # "kecepatan normal" untuk setiap waktu × sensor
print("v_ff (mph):", pd.Series(vff).describe().round(1).to_dict())
print("Profil historis:", PROFILE.shape, "(slot 15 menit × sensor)")

## 3. Feature engineering
Untuk setiap **waktu asal t** dan sensor, model melihat informasi yang **sudah tersedia di waktu t** saja:

| Kelompok | Fitur | Intuisi |
|---|---|---|
| Kondisi terkini | `v_now`, lag 5/15/30/60 menit | kemacetan punya "momentum" |
| Tren | rata-rata & std 30/60 menit terakhir | sedang memburuk atau membaik? |
| Pola rutin | `prof_target` (kecepatan normal di jam target), `v_yesterday`, `v_lastweek` | jam sibuk berulang tiap hari/minggu |
| **Deviasi** | `dev_now` = kecepatan sekarang − normal | **sinyal kejadian tidak rutin** (kecelakaan, cuaca) |
| Spasial | `nb_ratio_now` — kondisi 5 sensor terdekat di jalan & arah yang sama | macet merambat antar sensor |
| Kalender | jam, hari, akhir pekan, **hari libur AS** di waktu target | |
| Cuaca | hujan sekarang & 3 jam terakhir (Open-Meteo) | hujan memperlambat lalu lintas |
| Statis | jumlah lajur, v_ff, nomor jalan tol | karakter tiap ruas |

**Target:** model memprediksi **deviasi** `kecepatan(t+h) − profil(t+h)`, lalu ditambah kembali ke profil. Jadi model tidak perlu belajar ulang pola jam sibuk — ia fokus pada yang *tidak rutin*.

In [ ]:
S_pd = pd.DataFrame(S)
ROLL6  = S_pd.rolling(6,  min_periods=4).mean().values.astype("float32")
ROLL12 = S_pd.rolling(12, min_periods=8).mean().values.astype("float32")
STD12  = S_pd.rolling(12, min_periods=8).std().values.astype("float32")
R = (S / vff).astype("float32")                                               # speed ratio

# tetangga spasial: 5 sensor terdekat di jalan tol & arah yang sama (cadangan: terdekat mana pun)
lat, lon = meta.latitude.values, meta.longitude.values
D = np.hypot((lat[:, None] - lat[None]) * 111, (lon[:, None] - lon[None]) * 88)      # jarak kira-kira (km)
same = (meta.fwy.values[:, None] == meta.fwy.values[None]) & (meta.dir.values[:, None] == meta.dir.values[None])
np.fill_diagonal(D, np.inf)
NB = np.array([np.argsort(np.where(same[i], D[i], D[i] + 1e3))[:5] for i in range(N)])
R_NB = np.nanmean(R[:, NB], axis=2).astype("float32")
print("Jarak median ke 5 tetangga:", round(float(np.median(np.take_along_axis(D, NB, 1))), 2), "km")

HOLIDAYS = pd.to_datetime(["2017-01-02", "2017-01-16", "2017-02-20", "2017-05-29"])   # libur federal AS Jan–Jun 2017
print("Libur:", [d.strftime("%d %b") for d in HOLIDAYS])

### 3.1 Cuaca historis (Open-Meteo)
Diambil dari arsip Open-Meteo (gratis, tanpa API key, lisensi CC BY 4.0) untuk titik tengah area sensor. Kalau gagal (tidak ada internet), notebook **tetap jalan tanpa fitur cuaca** — kamu akan melihat pesannya.

In [ ]:
import requests
WEATHER_CSV = f"{OUT_DIR}/weather_open_meteo.csv"
weather = None
try:
    if os.path.exists(WEATHER_CSV):
        weather = pd.read_csv(WEATHER_CSV, parse_dates=["time"])
    else:
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {"latitude": round(float(np.mean(lat)), 3), "longitude": round(float(np.mean(lon)), 3),
                  "start_date": str(times.min().date()), "end_date": str(times.max().date()),
                  "hourly": "precipitation,temperature_2m", "timezone": "America/Los_Angeles"}
        js = requests.get(url, params=params, timeout=60).json()
        weather = pd.DataFrame(js["hourly"]); weather["time"] = pd.to_datetime(weather["time"])
        weather.to_csv(WEATHER_CSV, index=False)
    w = weather.set_index("time")["precipitation"]
    w = w[~w.index.duplicated()]
    RAIN = w.reindex(times.floor("h")).fillna(0).values.astype("float32")             # mm/jam pada jam waktu asal
    RAIN3 = pd.Series(w.rolling(3, min_periods=1).sum()).reindex(times.floor("h")).fillna(0).values.astype("float32")
    print(f"✅ Cuaca: {len(weather)} jam | total hujan {w.sum():.0f} mm | jam hujan: {(w > 0.1).sum()}")
except Exception as e:
    RAIN = RAIN3 = None
    print("⚠️  Cuaca tidak tersedia, lanjut tanpa fitur cuaca:", type(e).__name__, e)

In [ ]:
LANES, FWY = meta.lanes.values.astype("float32"), meta.fwy.values.astype("float32")
STEP_MIN = 5

def make_rows(O, h):
    '''O: indeks waktu asal, h: horizon (langkah 5 menit) → (X, y, info). Hanya memakai data ≤ waktu asal.'''
    O = np.asarray(O); O = O[O >= 2016]                       # butuh 1 minggu histori untuk v_lastweek
    n = len(O)
    t_target = times[O] + pd.Timedelta(minutes=STEP_MIN * h)
    tgt = O + h
    g  = lambda A, idx: A[idx].ravel()
    rep = lambda a: np.repeat(np.asarray(a, dtype="float32"), N)
    X = pd.DataFrame({
        "v_now": g(S, O), "v_lag5m": g(S, O - 1), "v_lag15m": g(S, O - 3), "v_lag30m": g(S, O - 6), "v_lag60m": g(S, O - 12),
        "v_roll30m": g(ROLL6, O), "v_roll60m": g(ROLL12, O), "v_std60m": g(STD12, O),
        "v_yesterday": g(S, tgt - 288), "v_lastweek": g(S, tgt - 2016),
        "prof_target": PROFILE[slot_of(t_target)].ravel(),
        "dev_now": g(S, O) - g(P, O), "ratio_now": g(R, O), "nb_ratio_now": g(R_NB, O),
        "hour_target": rep(t_target.hour + t_target.minute / 60), "dow_target": rep(t_target.dayofweek),
        "weekend_target": rep(t_target.dayofweek >= 5), "holiday_target": rep(t_target.normalize().isin(HOLIDAYS)),
        "lanes": np.tile(LANES, n), "vff": np.tile(vff, n), "fwy": np.tile(FWY, n),
    })
    if RAIN is not None:
        X["rain_now_mm"], X["rain_3h_mm"] = rep(RAIN[O]), rep(RAIN3[O])
    y = np.where((tgt < T)[:, None], S[np.minimum(tgt, T - 1)], np.nan).ravel() if n else np.array([])
    info = pd.DataFrame({"t_idx": np.repeat(O, N), "sensor_idx": np.tile(np.arange(N), n)})
    return X, y, info

X_demo, y_demo, _ = make_rows(np.array([20000]), 12)
print("Contoh satu waktu asal:", X_demo.shape, "(325 baris = 325 sensor)"); X_demo.head(3).round(2)

## 4. Membangun dataset train / validasi / test
Waktu asal diambil **setiap 15 menit** (bukan 5) — cukup rapat untuk belajar, dan 3× lebih hemat memori.

Satu model terpisah untuk setiap horizon: **h = 12 langkah (1 jam)** dan **h = 36 langkah (3 jam)**.

In [ ]:
HORIZONS = {"1h": 12, "3h": 36}
TEST_MODE = os.environ.get("NB_TEST") == "1"
MAX_TRAIN_ROWS = 300_000 if TEST_MODE else 2_500_000
O_all = np.arange(0, T, 3)
t_all = times[O_all]

datasets = {}
for name, h in HORIZONS.items():
    tgt_time = t_all + pd.Timedelta(minutes=STEP_MIN * h)
    parts = {"train": O_all[np.asarray(tgt_time < TRAIN_END)],
             "val":   O_all[np.asarray((t_all >= TRAIN_END) & (tgt_time < VAL_END))],
             "test":  O_all[np.asarray(t_all >= VAL_END) & (O_all + h < T)]}
    datasets[name] = {}
    for part, O in parts.items():
        X, y, info = make_rows(O, h)
        ok = ~np.isnan(y) & ~np.isnan(X["v_now"].values) & ~np.isnan(X["prof_target"].values)
        X, y, info = X[ok].reset_index(drop=True), y[ok], info[ok].reset_index(drop=True)
        if part == "train" and len(X) > MAX_TRAIN_ROWS:
            idx = np.random.default_rng(42).choice(len(X), MAX_TRAIN_ROWS, replace=False)
            X, y, info = X.iloc[idx].reset_index(drop=True), y[idx], info.iloc[idx].reset_index(drop=True)
        datasets[name][part] = (X, y, info)
        print(f"{name} {part:5s}: {len(X):>9,} baris")
FEATURES = list(datasets["1h"]["train"][0].columns)
print("\nJumlah fitur:", len(FEATURES))

## 5. Training Model B (LightGBM)
**LightGBM** = gradient boosting berbasis pohon yang sangat cepat untuk jutaan baris tabular — standar industri untuk forecasting tabular (sering menang di kompetisi Kaggle time series).

- Target = **deviasi dari profil**.
- *Early stopping* memakai data **validasi Mei**: berhenti menambah pohon ketika error Mei tidak membaik 50 putaran.

In [ ]:
import lightgbm as lgb
models = {}
for name in HORIZONS:
    (Xtr, ytr, _), (Xva, yva, _) = datasets[name]["train"], datasets[name]["val"]
    t0 = time.time()
    mdl = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=63, min_child_samples=200,
                            subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                            random_state=42, n_jobs=-1, verbose=-1)
    mdl.fit(Xtr, ytr - Xtr["prof_target"], eval_set=[(Xva, yva - Xva["prof_target"])], eval_metric="l1",
            categorical_feature=["fwy", "dow_target"], callbacks=[lgb.early_stopping(50, verbose=False)])
    models[name] = mdl
    print(f"Model {name}: {mdl.best_iteration_} pohon | {time.time() - t0:.0f} detik")

## 6. Evaluasi di data test (Juni 2017)
Tiga metode dibandingkan:
1. **Persistence** — "1 jam lagi sama dengan sekarang". Sulit dikalahkan untuk horizon pendek.
2. **Profil historis** — "sama dengan rata-rata hari & jam yang sama". Ini isi `bv_sensor_profile` dari batch layer.
3. **Model B**.

Metrik:
- **MAE** (mph) — rata-rata selisih kecepatan prediksi vs aktual.
- **Macro-F1 level** — setelah kecepatan diubah ke level Low/Moderate/High/Severe (ambang sama dengan notebook 01).
- **Recall macet** — dari semua kejadian High/Severe, berapa persen yang berhasil diprediksi? *Inilah metrik terpenting bagi city planner.*

Kita juga melihat subset **"tidak rutin"** (kondisi sekarang menyimpang > 10 mph dari normal) — di sinilah nilai sebenarnya sebuah model prediksi.

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score
LEVELS = ["Low", "Moderate", "High", "Severe"]
def to_level(speed, vff_rows):
    r = speed / vff_rows
    return np.select([r < 0.4, r < 0.6, r < 0.8], ["Severe", "High", "Moderate"], "Low")

rows, preds = [], {}
for name in HORIZONS:
    X, y, info = datasets[name]["test"]
    p_model = X["prof_target"].values + models[name].predict(X)
    methods = {"Persistence": X["v_now"].values, "Profil historis": X["prof_target"].values, "Model B": p_model}
    preds[name] = (X, y, info, methods)
    true_lvl = to_level(y, X["vff"].values); true_cong = np.isin(true_lvl, ["High", "Severe"])
    for subset, mask in [("Semua", np.ones(len(y), bool)), ("Tidak rutin", np.abs(X["dev_now"].values) > 10)]:
        for mname, p in methods.items():
            pl = to_level(p, X["vff"].values)
            rows.append({"horizon": name, "subset": subset, "metode": mname, "n": int(mask.sum()),
                         "MAE (mph)": np.mean(np.abs(p[mask] - y[mask])),
                         "RMSE (mph)": np.sqrt(np.mean((p[mask] - y[mask]) ** 2)),
                         "macro-F1 level": f1_score(true_lvl[mask], pl[mask], average="macro", labels=LEVELS, zero_division=0),
                         "recall macet": recall_score(true_cong[mask], np.isin(pl[mask], ["High", "Severe"]), zero_division=0),
                         "precision macet": precision_score(true_cong[mask], np.isin(pl[mask], ["High", "Severe"]), zero_division=0)})
evalres = pd.DataFrame(rows)
evalres.round(3).set_index(["horizon", "subset", "metode"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
pal = {"Persistence": "#9E9E9E", "Profil historis": "#EECA3B", "Model B": "#4C78A8"}
for ax, metric in zip(axes, ["MAE (mph)", "recall macet"]):
    d = evalres[evalres.subset == "Semua"].pivot(index="horizon", columns="metode", values=metric)[list(pal)]
    d.plot.bar(ax=ax, color=list(pal.values()), rot=0, width=0.75)
    ax.set_title(metric + " — data test Juni"); ax.set_xlabel("horizon"); ax.legend(fontsize=8)
    for c in ax.containers: ax.bar_label(c, fmt="%.2f", fontsize=7)
plt.tight_layout(); savefig("01_perbandingan_metode"); plt.show()

**Cara membaca:**
- Untuk **1 jam**, persistence biasanya sudah kuat — model harus mengalahkannya, terutama di **recall macet**.
- Untuk **3 jam**, persistence memburuk (kondisi sekarang kurang relevan), sedangkan profil makin penting. Model yang baik menggabungkan keduanya.
- Di subset **tidak rutin**, profil historis jatuh (ia tidak tahu ada kejadian), dan di sinilah fitur `dev_now`, tetangga spasial, dan cuaca bekerja.
- Hati-hati membaca **recall persistence** di subset tidak rutin: angkanya bisa tinggi karena ia terus "menyalin" kondisi macet, tetapi **precision-nya rendah** (banyak alarm palsu setelah macet sudah usai). Selalu baca recall **bersama** precision.

In [ ]:
X, y, info, methods = preds["1h"]
hot_sensor = sensors[np.argmax(np.nanmean(R[~is_train] < 0.6, axis=0))]          # sensor paling sering macet di periode uji
s_idx = int(np.where(sensors == hot_sensor)[0][0])
sel = info.sensor_idx.values == s_idx
tt = times[info.t_idx.values[sel]] + pd.Timedelta(hours=1)
daily_cong = pd.Series(y[sel] / vff[s_idx] < 0.6, index=tt).groupby(tt.date).mean()
day = pd.Timestamp(daily_cong.idxmax())
dm = (tt >= day) & (tt < day + pd.Timedelta(days=1))
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(tt[dm], y[sel][dm], color="k", lw=2, label="aktual")
ax.plot(tt[dm], methods["Profil historis"][sel][dm], color="#EECA3B", lw=1.5, label="profil historis")
ax.plot(tt[dm], methods["Persistence"][sel][dm], color="#9E9E9E", lw=1, ls="--", label="persistence")
ax.plot(tt[dm], methods["Model B"][sel][dm], color="#4C78A8", lw=2, label="Model B (prediksi 1 jam sebelumnya)")
ax.axhline(0.6 * vff[s_idx], color="#E45756", ls=":", label="batas macet (High)")
ax.set(title=f"Sensor {hot_sensor} ({meta.loc[hot_sensor, 'name']}, fwy {meta.loc[hot_sensor, 'fwy']} {meta.loc[hot_sensor, 'dir']}) — {day.date()}",
       ylabel="mph"); ax.legend(fontsize=8, ncol=3)
savefig("02_contoh_prediksi_sensor"); plt.show()

**Perhatikan:** garis persistence "terlambat" 1 jam (ia hanya menyalin kondisi lama), sedangkan Model B mencoba mengantisipasi kapan macet dimulai dan berakhir.

In [ ]:
imp = pd.DataFrame({h: pd.Series(models[h].booster_.feature_importance("gain"), index=FEATURES) for h in HORIZONS})
imp = (imp / imp.sum()).sort_values("1h")
ax = imp.plot.barh(figsize=(7, 6), color=["#4C78A8", "#F58518"])
ax.set_title("Pentingnya fitur (proporsi gain LightGBM)"); ax.set_xlabel("proporsi")
savefig("03_feature_importance"); plt.show()

## 7. Logika yang sama di Spark (skala cluster)
Di notebook ini fitur dihitung dengan NumPy karena 17 juta baris muat di RAM Colab. Untuk ribuan sensor × bertahun-tahun, data tidak muat di satu mesin — fitur lag dihitung dengan **Spark Window Functions**: `lag(...) OVER (PARTITION BY sensor ORDER BY waktu)`.

Sel ini membuktikan bahwa hasil Spark **sama persis** dengan NumPy untuk satu bulan data (opsional, ±2 menit).

In [ ]:
RUN_SPARK_DEMO = True
if RUN_SPARK_DEMO:
    if IN_COLAB and shutil.which("java") is None:
        !apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    %pip install -q pyspark==3.5.5
    from pyspark.sql import SparkSession, functions as F, Window
    spark = (SparkSession.builder.master("local[*]").appName("ModelB-features")
             .config("spark.driver.memory", "4g").config("spark.sql.session.timeZone", "UTC").getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    sdf = spark.read.parquet(local_master).filter(F.col("month") == 3)
    w = Window.partitionBy("sensor_id").orderBy("timestamp")
    feats_sp = (sdf.withColumn("v_lag60m", F.lag("speed", 12).over(w))
                   .withColumn("v_roll60m", F.avg("speed").over(w.rowsBetween(-11, 0))))
    s0 = int(sensors[0])
    sp = (feats_sp.filter(F.col("sensor_id") == s0).orderBy("timestamp").select("timestamp", "v_lag60m", "v_roll60m").toPandas())
    sp = sp.set_index("timestamp")
    npv = pd.DataFrame({"v_lag60m": pd.Series(S[:, 0], index=times).shift(12),
                        "v_roll60m": pd.Series(S[:, 0], index=times).rolling(12, min_periods=1).mean()}).reindex(sp.index)
    inner = sp.index[sp.index > sp.index.min() + pd.Timedelta(hours=2)]          # abaikan awal bulan (batas partisi) & jam DST
    inner = inner[~(npv.loc[inner].isna().any(axis=1) | sp.loc[inner].isna().any(axis=1))]
    print("Selisih maksimum Spark vs NumPy (lag 60 menit):", float(np.max(np.abs(sp.loc[inner, "v_lag60m"] - npv.loc[inner, "v_lag60m"]))))
    feats_sp.filter(F.col("sensor_id") == s0).select("timestamp", "sensor_id", "speed", "v_lag60m", "v_roll60m").show(5)

Selisih ≈ 0 (kecuali sekitar jam DST, karena Spark menghitung lag per baris sedangkan grid NumPy kita memiliki 12 slot kosong pada jam yang tidak ada). Artinya pipeline fitur bisa dipindah ke cluster tanpa mengubah logika.

## 8. Simpan model & prediksi untuk serving layer
Tiga keluaran untuk dashboard:
1. **Model** (`model_b_1h.joblib`, `model_b_3h.joblib`) + konfigurasi + profil + v_ff → backend bisa memprediksi kapan saja.
2. **`forecasts_june.parquet`** — prediksi setiap 15 menit untuk seluruh Juni → dipakai dashboard untuk **replay** seolah-olah real-time (simulasi stream).
3. **`latest_forecast.csv`** — prediksi dari titik waktu terakhir data, per sensor, lengkap dengan koordinat → langsung untuk peta.

In [ ]:
for name, mdl in models.items():
    joblib.dump(mdl, f"{OUT_DIR}/model_b_{name}.joblib")
pd.DataFrame(PROFILE, columns=sensors).rename_axis("slot15").to_parquet(f"{OUT_DIR}/profile_train.parquet")
pd.DataFrame({"sensor_id": sensors, "v_ff": vff}).to_csv(f"{OUT_DIR}/vff_train.csv", index=False)

# 2) prediksi Juni untuk replay dashboard
fc = []
for name in HORIZONS:
    X, y, info, methods = preds[name]
    fc.append(pd.DataFrame({"t_idx": info.t_idx, "sensor_idx": info.sensor_idx, f"pred_{name}": methods["Model B"].astype("float32"),
                            f"actual_{name}": y.astype("float32")}))
fc = fc[0].merge(fc[1], on=["t_idx", "sensor_idx"], how="outer")
fc["timestamp"] = times[fc.t_idx.values]; fc["sensor_id"] = sensors[fc.sensor_idx.values]
fc["speed_now"] = S[fc.t_idx.values, fc.sensor_idx.values]
vr = vff[fc.sensor_idx.values]
for col, src in [("level_now", "speed_now"), ("level_1h", "pred_1h"), ("level_3h", "pred_3h")]:
    fc[col] = np.where(fc[src].notna(), to_level(fc[src].values, vr), None)
fc = fc.drop(columns=["t_idx", "sensor_idx"]).sort_values(["timestamp", "sensor_id"])
fc.to_parquet(f"{OUT_DIR}/forecasts_june.parquet", index=False)

# 3) prediksi dari titik waktu terakhir
last = np.array([T - 1])
latest = pd.DataFrame({"sensor_id": sensors, "speed_now": S[-1], "level_now": to_level(S[-1], vff)})
for name, h in HORIZONS.items():
    Xl, _, _ = make_rows(last, h)
    p = Xl["prof_target"].values + models[name].predict(Xl[FEATURES])
    latest[f"pred_{name}"], latest[f"level_{name}"] = p, to_level(p, vff)
latest = latest.join(meta[["fwy", "dir", "name", "latitude", "longitude"]], on="sensor_id")
latest.insert(0, "as_of", times[-1]); latest.to_csv(f"{OUT_DIR}/latest_forecast.csv", index=False)

config = {"horizons_steps": HORIZONS, "step_minutes": STEP_MIN, "features": FEATURES, "target": "speed(t+h) - profile(t+h)",
          "level_thresholds_ratio": {"Severe": "<0.4", "High": "0.4-0.6", "Moderate": "0.6-0.8", "Low": ">=0.8"},
          "train": [str(times.min()), str(TRAIN_END)], "val": [str(TRAIN_END), str(VAL_END)], "test": [str(VAL_END), str(times.max())],
          "weather": RAIN is not None, "holidays": [str(d.date()) for d in HOLIDAYS],
          "best_iteration": {n: int(m.best_iteration_) for n, m in models.items()},
          "test_metrics": evalres.round(4).to_dict(orient="records")}
json.dump(config, open(f"{OUT_DIR}/model_b_config.json", "w"), indent=2, default=str)
print(f"forecasts_june.parquet: {len(fc):,} baris | latest_forecast.csv: {len(latest)} sensor (as of {times[-1]})")
for f in sorted(os.listdir(OUT_DIR)): print("  ", f)
latest.head()

## 9. Ringkasan
| Hasil | Lokasi |
|---|---|
| Model B (1 jam & 3 jam) | `output/04_model_b/model_b_1h.joblib`, `model_b_3h.joblib` |
| Profil & v_ff (dari data train) | `profile_train.parquet`, `vff_train.csv` |
| Prediksi Juni untuk replay dashboard | `forecasts_june.parquet` |
| Prediksi terbaru per sensor (untuk peta) | `latest_forecast.csv` |
| Konfigurasi + metrik lengkap | `model_b_config.json` |
| Grafik laporan | `figures/` |

**Poin untuk laporan:**
1. Split **kronologis**, v_ff & profil dari data **train saja** → tidak ada kebocoran.
2. Model dibandingkan dengan **dua baseline kuat**; lihat tabel di bagian 6 untuk angka finalmu.
3. Target **deviasi dari profil** membuat model fokus pada kemacetan tidak rutin.
4. Logika fitur terbukti setara di **Spark** → siap diskalakan ke cluster.

---
**Selanjutnya → Serving layer & dashboard:** backend API yang menggabungkan batch views (nb 01), Model A real-time dari video (nb 03), dan prediksi Model B (nb 04) → dashboard city planner.